In [42]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import MinMaxScaler
import joblib

# ---------- Step 1: Load CSV ----------
df = pd.read_csv("swap_log.csv")



In [43]:
# ---------- Step 2: Convert hex → int ----------
def hex_to_int(x):
    try:
        if isinstance(x, str) and x.startswith("0x"):
            return int(x, 16)
        else:
            return int(x)
    except:
        return 0
    

In [44]:
df["VA"] = df["VA"].apply(hex_to_int)
df["PFN"] = df["PFN"].apply(hex_to_int)
df["mapping"] = df["mapping"].apply(hex_to_int)

scaler = joblib.load('swap_scaler.joblib')
df[["VA", "PFN", "folio_index", "mapping", "latency_ns"]] = scaler.fit_transform(
    df[["VA", "PFN", "folio_index", "mapping", "latency_ns"]]
)

In [45]:
df["VA"] = df["VA"].apply(hex_to_int)
df["PFN"] = df["PFN"].apply(hex_to_int)
df["mapping"] = df["mapping"].apply(hex_to_int)

scaler = joblib.load('swap_scaler.joblib')
df[["VA", "PFN", "folio_index", "mapping", "latency_ns"]] = scaler.fit_transform(
    df[["VA", "PFN", "folio_index", "mapping", "latency_ns"]]
)

SEQ_LEN = 10
process_data_sqen = {}
for pid in df["PID"].unique():
    pid_data = df[df["PID"] == pid][["VA", "PFN", "folio_index", "mapping", "latency_ns"]].values
    n = len(pid_data)
    for i in range(n):
        start_idx = max(0, i - SEQ_LEN)
        seq = pid_data[start_idx:i]
        # pad if sequence smaller than SEQ_LEN
        if len(seq) < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - len(seq), pid_data.shape[1]))
            seq = np.vstack((pad, seq))

    

In [51]:

import numpy as np
from collections import defaultdict

class ProcessSeqStore:
    """
    Maintain per-PID fixed-length swap fault sequences (LIFO style).

    Each process (pid) → np.ndarray of shape (seq_len, feature_dim)
    """

    def __init__(self, seq_len: int = 10, feature_dim: int = 5):
        self.seq_len = seq_len
        self.feature_dim = feature_dim
        self.store = defaultdict(lambda: np.zeros((seq_len, feature_dim), dtype=np.float64))

    def update(self, pid: int, new_entry: np.ndarray):
        """
        Update sequence for a specific PID with a new entry.

        Parameters
        ----------
        pid : int
            Process ID.
        new_entry : np.ndarray
            1D array (feature_dim,) representing the new swap fault entry.
        """
        if new_entry.ndim != 1 or new_entry.shape[0] != self.feature_dim:
            raise ValueError(f"new_entry shape must be ({self.feature_dim},), got {new_entry.shape}")

        seq = self.store[pid]

        # Append new entry → keep last N (LIFO/FIFO style)
        seq = np.vstack([seq, new_entry[np.newaxis, :]])
        if seq.shape[0] > self.seq_len:
            seq = seq[-self.seq_len:, :]  # drop oldest if overflow
        elif seq.shape[0] < self.seq_len:
            pad = np.zeros((self.seq_len - seq.shape[0], self.feature_dim))
            seq = np.vstack([pad, seq])

        self.store[pid] = seq

    def get(self, pid: int) -> np.ndarray:
        """Return the current sequence for the given PID (zeros if new)."""
        return self.store[pid]

    def all_pids(self):
        """List all tracked PIDs."""
        return list(self.store.keys())

    def __len__(self):
        return len(self.store)



In [ ]:
# create store for 15 processes, seq length 10, feature_dim 5
seq_store = ProcessSeqStore(seq_len=10, feature_dim=5)

# add new entry for PID 1752
new_entry = np.array([0.65, 0.47, 0.0087, 0.0431, 0.1177])

seq_store.update(1752, new_entry)


# another process
seq_store.update(1801, np.array([0.81, 0.56, 0.0098, 0.0521, 0.1464]))

# get sequence
print(seq_store.get(1752))
print(seq_store.all_pids())  # → [1752, 1801]


[[0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]
 [0.65   0.47   0.0087 0.0431 0.1177]]
[1752, 1801]


In [ ]:
X=[]
for pid in df["PID"].unique():
    pid_data = df[df["PID"] == pid][["VA", "PFN", "folio_index", "mapping", "latency_ns"]].values
    n = len(pid_data)
    
    # Iterate over each entry as target
    for i in range(n):
        start_idx = max(0, i - SEQ_LEN)
        seq = pid_data[start_idx:i]

        # pad if sequence smaller than SEQ_LEN
        if len(seq) < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - len(seq), pid_data.shape[1]))
            seq = np.vstack((pad, seq))
        X.append(seq)

df['past_seq'] = X



In [55]:
from tensorflow.keras.models import load_model
model = load_model("swap_lstm_model.h5")

# Save in modern Keras format
model.save("swap_lstm_model.keras")

TypeError: Could not locate function 'mse'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.metrics', 'class_name': 'function', 'config': 'mse', 'registered_name': 'mse'}

In [56]:
from tensorflow.keras.models import load_model

# Load the legacy model in its original environment
model = load_model("swap_lstm_model.h5")

# Save in the new official .keras format
model.save("swap_lstm_model.keras")


TypeError: Could not locate function 'mse'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.metrics', 'class_name': 'function', 'config': 'mse', 'registered_name': 'mse'}

In [57]:
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import MeanAbsoluteError

# ---- custom object mapping for legacy models ----
custom_objects = {
    "mse": MeanSquaredError(),
    "mae": MeanAbsoluteError()
}

# ---- Load model safely ----
print("🔄 Loading legacy model (ignoring compile settings)...")
model = load_model("swap_lstm_model.h5", compile=False, custom_objects=custom_objects)

print("✅ Model loaded successfully. Now saving in modern Keras format...")
model.save("swap_lstm_model.keras")

print("🎉 Conversion done! Saved as swap_lstm_model.keras")


🔄 Loading legacy model (ignoring compile settings)...
✅ Model loaded successfully. Now saving in modern Keras format...
🎉 Conversion done! Saved as swap_lstm_model.keras


In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
import tensorflow as tf

print("🔧 Rebuilding model architecture manually...")

# rebuild same architecture
model = Sequential([
    Input(shape=(10, 5)),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(5, activation='linear')
])

print("✅ Model architecture rebuilt.")

# now load weights from your old model
try:
    model.load_weights("swap_lstm_model.h5")
    print("✅ Weights loaded successfully from swap_lstm_model.h5")
except Exception as e:
    print("⚠️ Could not load weights:", e)

# save it cleanly in new format
model.save("swap_lstm_model_fixed.keras")
print("💾 Saved clean model as swap_lstm_model_fixed.keras")


🔧 Rebuilding model architecture manually...
✅ Model architecture rebuilt.
⚠️ Could not load weights: Layer count mismatch when loading weights from file. Model expected 4 layers, found 3 saved layers.
💾 Saved clean model as swap_lstm_model_fixed.keras
